# Banking email routing (v2)

A short customer message, such as "my card has not arrived", belongs to one of 77 banking topics. This notebook trains three small models with Pecca and keeps the best one.

Run the cells from top to bottom. The raw files are already in `bank/raw_dataset/`.

## Setup

Pecca saves the trained model under `PECCA_HOME`.

In [ ]:
import os

ROOT = "<your project folder>"
if not os.path.isdir(os.path.join(ROOT, "bank", "raw_dataset")):
    ROOT = os.getcwd()

os.environ["PECCA_HOME"] = os.path.join(ROOT, ".pecca")
os.makedirs(os.environ["PECCA_HOME"], exist_ok=True)
print("Project folder:", ROOT)

Project folder: /Users/fazil/Desktop/pecca-core/testing/banking_email_routing


## Which device will we train on?

`cuda` is an NVIDIA GPU (a Windows PC, Colab, Databricks, or AWS). `mps` is the Apple GPU. If neither is available, we use the CPU.

In [16]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

import pecca.candidates.transformers_ft as tft
from sentence_transformers import SentenceTransformer

# xlmr_finetune asks Pecca for a device. Use the one we just chose.
tft._device = lambda: device

# e5_logreg embeds text with SentenceTransformer. Send that model to the GPU when we have one.
if device != "cpu":
    _original_init = SentenceTransformer.__init__

    def _init(self, *args, **kwargs):
        kwargs.setdefault("device", device)
        return _original_init(self, *args, **kwargs)

    SentenceTransformer.__init__ = _init

Device: mps


## Load and clean

Pecca needs four columns: `input` (the message), `llm_output` (the topic), `human_label` (empty here), and `call_name` (the job name).

In [17]:
import pandas as pd
import pecca
from pecca.data.dataset import Dataset, normalise_output
from sklearn.metrics import accuracy_score, f1_score

raw_dir = os.path.join(ROOT, "bank", "raw_dataset")
train_raw = pd.read_csv(os.path.join(raw_dir, "train.csv"))
test_raw = pd.read_csv(os.path.join(raw_dir, "test.csv"))

def prepare(raw):
    table = raw.copy()
    table.columns = [str(name).strip() for name in table.columns]
    table["text"] = table["text"].astype(str).str.strip()
    table["category"] = table["category"].astype(str).str.strip()
    keep = (table["text"] != "") & (~table["category"].isin(["", "nan", "None"]))
    table = table[keep]
    return pd.DataFrame({
        "input": table["text"].to_numpy(),
        "llm_output": table["category"].to_numpy(),
        "human_label": None,
        "call_name": "route_banking_email",
        "language": "en",
    }).reset_index(drop=True)

train_ready = prepare(train_raw)
test_ready = prepare(test_raw)
print("train rows:", len(train_ready))
print("test rows:", len(test_ready))
train_ready.head()

train rows: 10003
test rows: 3080


,input,llm_output,human_label,call_name,language
0,I am still waiting on my card?,card_arrival,None,route_banking_email,en
1,What can I do if my card still hasn't arrived ...,card_arrival,None,route_banking_email,en
2,I have been waiting over a week. Is the card s...,card_arrival,None,route_banking_email,en
3,Can I track my card while it is in the process...,card_arrival,None,route_banking_email,en
4,"How do I know if I will get my card, or if it ...",card_arrival,None,route_banking_email,en


## Train

Pecca trains `tfidf_linear`, `e5_logreg`, and `xlmr_finetune` on the same slices of the training file. It keeps the one with the best macro F1. The word model stays on the CPU. The other two use the device printed above.

In [21]:
train_ds = Dataset(train_ready, source="train.csv")
print(pecca.profile(train_ds, call="route_banking_email").table())

result = pecca.train(
    "route_banking_email",
    dataset=train_ds,
    candidates=[
        "tfidf_linear",
        "e5_logreg"
    ],
    progress=print,
)

leaderboard = pd.DataFrame(result.leaderboard)
print("winner:", result.winner)
leaderboard

route_banking_email classification  77 labels     10,003 rows  replaceable ✔
  tfidf_linear: macro_f1 0.8668
  e5_logreg: macro_f1 0.8723
winner: e5_logreg


,candidate,status,metric,metric_std,fit_time_s,predict_latency_ms,winner
0,tfidf_linear,ok,0.866831,0.006239,1.774984,0.688042,NaN
1,e5_logreg,ok,0.872344,0.005197,0.852020,0.133667,True


## Test the winner

The official test file was not used for training. Accuracy is the share of messages with the right topic. Macro F1 averages that quality across all 77 topics, so a rare topic counts the same as a common one.

In [19]:
model = pecca.load(result.model_path)
predictions = model.predict(test_ready["input"].tolist())

y_true = [normalise_output(label) for label in test_ready["llm_output"]]
y_pred = [normalise_output(item.label) for item in predictions]

print(f"test accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"test macro F1: {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")

test accuracy: 0.8883
test macro F1: 0.8868


## Try a message

Enter a `question`, then run the cell. The saved winner predicts the topic.

In [20]:
question = "I am still waiting on my card. Where is it?"

answer = pecca.load(result.model_path).predict([question])[0]
print("Message:", question)
print("Predicted category:", answer.label)
if answer.confidence is not None:
    print(f"Confidence: {answer.confidence:.0%}")

Message: I am still waiting on my card. Where is it?
Predicted category: card_arrival
Confidence: 83%


In [22]:
question = "I am still waiting on my card. Where is it?"
answer = model.predict([question])[0]

print(answer.label)
print(f"Confidence: {answer.confidence:.0%}")
print("Fallback:", answer.fallback)


card_arrival
Confidence: 83%
Fallback: False
